# Software gestionale per un negozio di prodotti vegani

Questo notebook ha lo scopo di presentare il prototipo del software gestionale richiesto da **BioMarket sas**, un punto vendita di prodotti vegani.

Il software richiesto dall'azienda cliente doveva contenere le seguenti funzionalità:
- Una funzione di guida all'utilizzo per l'utente;
- Funzioni di elenco e salvataggio dell'inventario;
- Registrazione di acquisti e vendite dal magazzino;
- Calcolo e stampa dei profitti lordi e netti.

Il programma richiesto, inoltre, doveva essere utilizzato da riga di comando e i dati inseriti dovevano permanere anche dopo la chiusura del programma.

Ciascuna delle funzionalità richieste è argomento di ogni sezione di questo notebook insieme alle strutture dati e alle funzioni di supporto al programma. L'ultima sezione racchiuderà le funzioni costruite in un unico blocco di codice e simulerà degli esempi applicativi del programma.

## Il menu di aiuto e la validazione dei comandi
<a name="aiuto">

Questa sezione sarà dedicata al menu di aiuto e alla validazione dei comandi inseriti dall'utente.

Dal momento che i comandi rimarranno gli stessi nel corso del programma, la struttura dati scelta per elencarli è una tupla di tuple, dove il primo elemento è il comando accettato e il secondo ne spiega la funzione. In un set, poi, sono stati salvati i soli nomi dei comandi.

In [1]:
available_commands = (
    ("aggiungi", "aggiungi un prodotto al magazzino"),
    ("elenca", "elenca i prodotti in magazzino"),
    ("vendita", "registra una vendita effettuata"),
    ("profitti", "mostra i profitti totali"),
    ("aiuto", "mostra i possibili comandi"),
    ("chiudi", "esci dal programma")
)

available_commands_set = {command[0] for command in available_commands}

La funzione *ask_help* stampa a video i comandi disponibili e le loro funzionalità.

In [2]:
def ask_help():

  """
  ask_help: Stampa a video dei comandi disponibili e delle loro funzionalità.
  """

  print("I comandi disponibili sono i seguenti:")
  for command, description in available_commands:
    print(f"{command}: {description}")

La funzione *validate_command*, di supporto all'intero programma, prende in ingresso il comando inserito dall'utente e lo restituisce insieme a un booleano che attesta la sua validità.

In [3]:
def validate_command(command):

  """
  validate_command: Valida il comando e, in caso di errore, stampa la guida all'utilizzo

    Parameters
      command (str): Comando inserito dall'utente

    Returns
      command (str): Comando inserito dall'utente
      valid_command (bool): Booleano che verifica se il comando inserito è valido
  """

  is_valid = command in available_commands_set

  if not is_valid:
    print(f"Comando non valido")
    ask_help()

  return command, is_valid

## Il file *inventory.json*
<a name="inventario">

Questa sezione è dedicata alle funzioni di supporto per leggere e scrivere il file contenente i dati sull'inventario.

Per manipolare l'inventario all'interno del programma è stato scelto un dizionario dove le chiavi sono i nomi dei prodotti, mentre i valori sono dizionari in cui sono memorizzati quantità e prezzi di vendita e di acquisto. Di conseguenza, il formato scelto per salvare su disco questi dati è il JSON.

Premesso questo, la funzione *read_inventory* controlla se il file esiste e legge i dati al suo interno, altrimenti (come ad esempio in caso di primo utilizzo del programma) restituisce un dizionario vuoto.

In [4]:
import os
import json

INVENTORY_NAME_FILE = "inventory.json"

def read_inventory():

  """
  read_inventory: Legge il file inventory.json e restituisce un dizionario

    Returns
      inventory (dict): Dizionario con i dati dell'inventario (vuoto se il file non esiste)
  """

  inventory = {}

  if os.path.isfile(INVENTORY_NAME_FILE):
    with open(INVENTORY_NAME_FILE) as inventory_file:
      inventory = json.load(inventory_file)

  return inventory

La funzione *write_inventory*, che prende in ingresso l'inventario, scrive i dati modificati dal programma nel file *inventory.json*.

In [5]:
def write_inventory(inventory):

  """
  write_inventory: Scrive il dizionario inventory nel file inventory.json

    Parameters
      inventory (dict): Dizionario con i dati dell'inventario
  """

  with open(INVENTORY_NAME_FILE, "w") as inventory_file:
    json.dump(inventory, inventory_file, indent=4)

## Operazioni di lettura dell'inventario
<a name="lettura">

Questa sezione è dedicata alle funzionalità per la lettura e la sintesi dei dati di inventario. Ricordiamo che, su richiesta di BioMarket sas, queste funzionalità riguardano l'elenco dei prodotti con le quantità e i prezzi disponibili e il calcolo dei profitti lordi e netti.

La funzione *print_inventory* risponde alla prima richiesta stampando come tabella il dizionario *inventory* in ingresso. Qualora l'inventario fosse vuoto (per esempio, in caso di primo utilizzo), il programma stampa un avviso.

In [6]:
def print_inventory(inventory):

  """
  print_inventory: Stampa a video i dati dell'inventario in formato tabulare

    Parameters
      inventory (dict): Dizionario con i dati dell'inventario
  """

  if len(inventory) == 0:
    print("Non sono disponibili dati di inventario")
  else:
    print("NOME\tQUANTITA\tPREZZO")
    for product in inventory:
      print(f"{product}\t{inventory[product]['quantity']}\t{inventory[product]['sell_price']:.2f}")

In modo analogo, la funzione *profits* calcola e stampa i profitti lordi (da intendersi come il totale dei prezzi di vendita) e netti (la differenza tra i profitti lordi e il totale dei prezzi di acquisto). Anche in questo caso, la funzione stampa un mesaggio di errore se l'inventario è vuoto.

In [7]:
def profits(inventory):

  """
  profits: Calcola e stampa i profitti lordi e netti dai dati di inventario

    Parameters
      inventory (dict): Dizionario con i dati dell'inventario
  """

  if len(inventory) == 0:
    print("Non sono disponibili dati per il calcolo dei profitti")
  else:
    total_gross = 0.0
    total_costs = 0.0
    for product in inventory:
      total_gross+= (inventory[product]['sell_price'] * inventory[product]['quantity'])
      total_costs += (inventory[product]['buy_price'] * inventory[product]['quantity'])

    print(f"Profitto lordo: € {total_gross:.2f}")
    print(f"Profitto netto: € {(total_gross - total_costs):.2f}")

## Operazioni di scrittura dell'inventario
<a name="scrittura">

Questa sezione è dedicata alle operazioni di scrittura dell'inventario, ossia la registrazione degli acquisti dei nuovi prodotti e delle vendite di quelli disponibili.

Il primo passo è stato definire le funzioni di supporto *validate_quantity* e *validate_price*. La prima funzione si assicura che le quantità inserite siano numeri interi positivi, mentre la seconda si assicura che i prezzi siano numeri decimali positivi.

In [8]:
def validate_quantity():

  """
  validate_quantity: Valida la quantità inserita dall'utente

    Returns
      quantity (int): Quantità inserita dall'utente
      valid_quantity (bool): Booleano che verifica se la quantità è valida
  """

  try:
    quantity = int(input("Inserire la quantità: "))
    assert quantity >= 0
    valid_quantity = True
  except ValueError:
    print("La quantità deve essere un numero intero, riprova")
    quantity = 0
    valid_quantity = False
  except AssertionError:
    print("La quantità deve essere un numero positivo, riprova")
    quantity = 0
    valid_quantity = False
  finally:
    return quantity, valid_quantity

In [9]:
def validate_price(price_type):

  """
  validate_price: Valida il prezzo di acquisto o vendita inserito dall'utente

    Parameters
      price_type (str): Indica se il prezzo è di acquisto o vendita

    Returns
      price (float): Prezzo inserito dall'utente
      valid_price (bool): Booleano che verifica se il prezzo è valido
  """

  try:
    price = float(input(f"Inserire il prezzo di {price_type}: "))
    assert price >= 0
    valid_price = True
  except ValueError:
    print(f"Il prezzo di {price_type} deve essere un numero, riprova")
    price = 0
    valid_price = False
  except AssertionError:
    print(f"Il prezzo di {price_type} deve essere un numero positivo, riprova")
    price = 0
    valid_price = False
  finally:
    return price, valid_price

Premesso questo, è stata definita la funzione *aggiungi* che prende in ingresso l'inventario e lo aggiorna con gli acquisti. All'interno della funzione è stato inserito un controllo per validare la risposta in caso di aggiunta di prodotti multipli.

In [10]:
def aggiungi(inventory):

  """
  aggiungi: Registra gli acquisti e aggiorna l'inventario

    Parameters
      inventory (dict): Dizionario con i dati correnti dell'inventario

    Returns
      inventory (dict): Dizionario con i dati aggiornati dell'inventario
  """

  answer = ""

  while answer != "no":
    answer = ""
    name = input("Nome del prodotto: ")

    if not name in inventory:
      valid_input = False
      while not valid_input:
        buy_price, valid_input = validate_price("acquisto")

      valid_input = False
      while not valid_input:
        sell_price, valid_input = validate_price("vendita")

      inventory[name] = {
        "quantity": 0,
        "buy_price": buy_price,
        "sell_price": sell_price
      }

    valid_input = False
    while not valid_input:
      quantity, valid_input = validate_quantity()

    inventory[name]["quantity"] += quantity

    print(f"AGGIUNTO: {quantity} X {name}")

    while not answer in ("si", "no"):
      answer = input("Vuoi aggiungere un altro prodotto? (si/no): ").lower()
      if not answer in ("si", "no"):
        print("La risposta inserita non è valida, riprova")

  return inventory

In modo analogo è stata definita la funzione *vendita*, in cui però è stato inserito un controllo aggiuntivo sulla disponibilità di prodotto in termine di nome e quantità.

In [16]:
def vendita(inventory):

  """
  vendita: Registra le vendite e aggiorna l'inventario

    Parameters
      inventory (dict): Dizionario con i dati correnti dell'inventario

    Returns
      inventory (dict): Dizionario con i dati aggiornati dell'inventario
  """

  answer = ""
  sold_items = []
  total_sale = 0

  while answer != "no":
    answer = ""
    name = input("Nome del prodotto: ")

    if not name in inventory:
      print(f"Il prodotto {name} non è presente in magazzino")
    else:
      valid_input = False
      while not valid_input:
        quantity, valid_input = validate_quantity()

      if inventory[name]["quantity"] < quantity:
        print(f"Il prodotto {name} non è disponibile nella quantità inserita")
      else:
        inventory[name]["quantity"] -= quantity
        total_sale += inventory[name]["sell_price"] * quantity
        sold_items.append({"name": name, "qty": quantity, "price": inventory[name]["sell_price"]})
        print(f"VENDUTO: {quantity} X {name}")


    while not answer in ("si", "no"):
      answer = input("Vuoi aggiungere un altro prodotto? (si/no): ").lower()
      if not answer in ("si", "si"):
        print("La risposta inserita non è valida, riprova")

  if sold_items:
      print("VENDITA REGISTRATA")
      for item in sold_items:
          print(f"- {item['qty']} X {item['name']}: €{item['price']:.2f}")
      print(f"Totale: €{total_sale:.2f}")

  return inventory

## Corpo del programma ed esempi applicati
<a name="applicazione">

Concludiamo il progetto racchiudendo all'interno della funzione *main* tutte le procedure descritte nelle sezioni precedenti.

Dopo aver riportato data e ora della sessione di lavoro, il programma valida il comando e ne esegue la funzione corrispondente finché l'utente non inserisce il comando di uscita. Al termine della sessione di lavoro, il programma salva l'inventario nel file JSON *inventory.json* e riporta data e ora della fine della sessione di lavoro.

In [12]:
import datetime as dt

def main():

  """
  main: Corpo del programma
  """

  inventory = read_inventory()
  command = ""

  print(f"Inizio sessione di lavoro: {dt.datetime.now()}")
  print("Bentornato!")

  while command!="chiudi":
    command, valid_command = validate_command(input("Inserisci un comando: ").lower())

    if valid_command:
      if command == "aiuto":
        ask_help()
      elif command == "elenca":
        print_inventory(inventory)
      elif command == "vendita":
        inventory = vendita(inventory)
      elif command == "aggiungi":
        inventory = aggiungi(inventory)
      elif command == "profitti":
        profits(inventory)
      elif command == "chiudi":
        write_inventory(inventory)
        print("Bye bye!")

      print("\n", end="")

  print(f"Fine sessione di lavoro: {dt.datetime.now()}")

A dimostrazione dell'utilizzo del programma, si simuleranno due sessioni di lavoro.

Nella prima sessione si svolgerà quanto segue:
1. Si aprirà la guida al programma;
2. Si aggiungeranno 20 unità di latte di soia (€ 0.80, € 1.40), 10 unità di tofu (€ 2.20, € 4.19) e 5 unità di seitan (€ 3.00, 5.49);
3. Si stamperà l'inventario;
4. Si venderanno 10 latte di soia, 10 seitan (non disponibili) e 5 tofu;
5. Si stamperanno i profitti;
6. Si esce dal programma.

In [13]:
main()

Inizio sessione di lavoro: 2026-03-17 14:26:59.877438
Bentornato!
Inserisci un comando: aiuto
I comandi disponibili sono i seguenti:
aggiungi: aggiungi un prodotto al magazzino
elenca: elenca i prodotti in magazzino
vendita: registra una vendita effettuata
profitti: mostra i profitti totali
aiuto: mostra i possibili comandi
chiudi: esci dal programma

Inserisci un comando: aggiungi
Nome del prodotto: latte di soia
Inserire il prezzo di acquisto: 0.8
Inserire il prezzo di vendita: 1.4
Inserire la quantità: 20
AGGIUNTO: 20 X latte di soia
Vuoi aggiungere un altro prodotto? (si/no): si
Nome del prodotto: tofu
Inserire il prezzo di acquisto: 2.2
Inserire il prezzo di vendita: 4.19
Inserire la quantità: 10
AGGIUNTO: 10 X tofu
Vuoi aggiungere un altro prodotto? (si/no): si
Nome del prodotto: seitan
Inserire il prezzo di acquisto: 3
Inserire il prezzo di vendita: 5.49
Inserire la quantità: 5
AGGIUNTO: 5 X seitan
Vuoi aggiungere un altro prodotto? (si/no): no

Inserisci un comando: elenca
NOME

Nella seconda sessione si svolgerà quanto segue:
1. Si stamperà l'inventario;
2. Si aggiungeranno 10 unità di seitan e 10 unità di burger di spinaci prima ai prezzi non corretti (€ 3.49, - € 5.99) e poi (€ 3.49, € 5.99)
3. Si venderanno 10 unità di seitan, 5 burger di spinaci e 5 latte di avena (non presenti);
4. Si stamperanno i profitti;
5. Si tenterà di eseguire il comando 'sconto' (non disponibile);
6. Si esce dal programma.

In [14]:
main()

Inizio sessione di lavoro: 2026-03-17 14:30:23.053683
Bentornato!
Inserisci un comando: elenca
NOME	QUANTITA	PREZZO
latte di soia	10	1.40
tofu	5	4.19
seitan	5	5.49

Inserisci un comando: aggiungi
Nome del prodotto: seitan
Inserire la quantità: 10
AGGIUNTO: 10 X seitan
Vuoi aggiungere un altro prodotto? (si/no): si
Nome del prodotto: burger di spinaci
Inserire il prezzo di acquisto: 3.49
Inserire il prezzo di vendita: -5.99
Il prezzo di vendita deve essere un numero positivo, riprova
Inserire il prezzo di vendita: 5.99
Inserire la quantità: 10
AGGIUNTO: 10 X burger di spinaci
Vuoi aggiungere un altro prodotto? (si/no): no

Inserisci un comando: vendita
Nome del prodotto: seitan
Inserire la quantità: 10
VENDUTO: 10 X seitan
Vuoi aggiungere un altro prodotto? (si/no): si
La risposta inserita non è valida, riprova
Nome del prodotto: burger di spinaci
Inserire la quantità: 5
VENDUTO: 5 X burger di spinaci
Vuoi aggiungere un altro prodotto? (si/no): si
La risposta inserita non è valida, ripr

La cella seguente stampa l'inventario ottenuto al termine delle due simulazioni.

In [15]:
print_inventory(read_inventory())

NOME	QUANTITA	PREZZO
latte di soia	10	1.40
tofu	5	4.19
seitan	5	5.49
burger di spinaci	5	5.99


## Conclusioni

Il progetto presentato in questo notebook è il prototipo di un software gestionale richiesto da **BioMarket sas**, un punto vendita di prodotti vegani.

[Nella prima sezione](#aiuto) sono state mostrate le funzioni per stampare la guida all'utilizzo e per validare i comandi inseriti dall'utente.

[Nella seconda sezione](#inventario) è stata scelta la struttura dati per memorizzare le informazioni sull'inventario e sono state definite due funzioni per leggere e scrivere nel file dedicato.

[Nella terza sezione](#lettura) sono state definite le due funzioni per la lettura e la sintesi dei dati di inventario. In particolare, la prima funzione stampa a video la tabella con le informazioni su quantità e prezzi dei prodotti disponibili, mentre la seconda calcola e stampa a video i profitti lordi e netti.

[Nella quarta sezione](#scrittura) sono state definite le funzioni per scrivere e modificare l'inventario. La prima funzione registra gli acquisti per il magazzino, mentre la seconda registra le vendite. Entrambe le funzioni si appoggiano a procedure di supporto per validare sia le quantità che i prezzi inseriti e restituiscono l'inventario aggiornato con tutte le modifiche.

[Nella quinta e ultima sezione](#applicazione), poi, le funzioni descritte sono state riunite in un'unica funzione dove il programma chiede i comandi all'utente finché questi non decide di uscire. A titolo di esempio, sono state simulate due sessioni di lavoro ed è stato stampato l'inventario ottenuto per validare i risultati del progetto.